# 01 · Lung atlas: donor filtering and doublet detection

First step of the healthy-lung pipeline (paper Methods: *Non-malignant lung single cell data processing*). This notebook processes all donors in the atlas; healthy samples are selected in File 02.

**Input**: full Human Lung Cell Atlas (HLCA core + extension, 2,282,447 cells), Sikkema *et al.* 2023, from [CELLxGENE](https://cellxgene.cziscience.com/collections/6f6d381a-7701-4781-935c-db10d30de293). The version used is `2b415299-371a-4e95-8cba-8c6036a72ad5` (released 14 April 2025), downloadable at https://datasets.cellxgene.cziscience.com/2b415299-371a-4e95-8cba-8c6036a72ad5.h5ad (~22 GB).

**Steps**
1. Keep donors with ≥ 100 cells and switch to the raw count matrix.
2. Score doublets with Scrublet (`sc.pp.scrublet`), run separately for each donor (`batch_key="donor_id"`) with 15 principal components.

**Output**: `intermediate_files/lung/01_lung_sikkema_atlas_raw_normal.h5ad` (raw counts and doublet scores, all donors). Doublets are removed and healthy samples selected in File 02.

## Setup

In [1]:
import os
import gc

import anndata
import scanpy as sc

In [2]:
# Paths: edit to match your environment
HLCA_H5AD = "/path/to/Data_Sikkema_2023_lung/2b415299-371a-4e95-8cba-8c6036a72ad5.h5ad"  # full HLCA (core + extension) from CELLxGENE
OUT_DIR = "intermediate_files/lung"

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load the atlas and keep donors with ≥ 100 cells

In [3]:
adata = sc.read(HLCA_H5AD)

sample_counts = adata.obs['donor_id'].value_counts()               # number of cells per donor
valid_samples = sample_counts[sample_counts >= 100].index          # donors with at least 100 cells
adata = adata[adata.obs['donor_id'].isin(valid_samples)].copy()

adata = anndata.AnnData(adata.raw.X, obs=adata.obs, var=adata.raw.var)  # use the raw count matrix
gc.collect()

3381

## 2. Doublet detection with Scrublet

In [4]:
sc.pp.scrublet(adata, batch_key="donor_id", n_prin_comps=15)  # doublet scores per donor
gc.collect()

10835

## 3. Save

In [5]:
adata.write(f'{OUT_DIR}/01_lung_sikkema_atlas_raw_normal.h5ad')